In [1]:
!pip install -q -U \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.7 MB/s eta 0:00:00


In [2]:
import transformers
import accelerate
import peft
import bitsandbytes
import trl
import tokenizers

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)
print("tokenizers:", tokenizers.__version__)

transformers: 5.17.0
accelerate: 1.15.0
peft: 0.20.0
bitsandbytes: 0.50.2
trl: 1.13.0
tokenizers: 0.23.1


In [3]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


In [4]:
dataset = load_dataset("databricks/databricks-dolly-15k")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [5]:
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})


In [6]:
print(dataset["train"][0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [7]:
print(dataset["train"].column_names)

['instruction', 'context', 'response', 'category']


In [8]:
example = dataset["train"][0]

print("Instruction:", example["instruction"])
print("Context:", example["context"])
print("Response:", example["response"])

Instruction: When did Virgin Australia start operating?
Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.
Response: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [9]:
def format_example(example):
    return {
        "text": f"""User: {example['instruction']}

Context: {example['context']}

Assistant: {example['response']}"""
    }

formatted_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [10]:
formatted_dataset

Dataset({
    features: ['instruction', 'context', 'response', 'category', 'text'],
    num_rows: 15011
})

In [11]:
print(formatted_dataset[0]["text"])

User: When did Virgin Australia start operating?

Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

Assistant: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [13]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("GEMMA_API")
login(token=token)

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


In [15]:
model_name = "google/gemma-3-1b-it"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [16]:
print(model.dtype)

torch.bfloat16


In [17]:
print(model.get_memory_footprint() / 1024**2, "MB")

908.9768085479736 MB


In [18]:
print(model.config.quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [20]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [21]:
print(tokenized_dataset[0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'text': "User: When did Virgin Australia start operating?\n\nContext: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 3

In [22]:
lora_config= LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

In [23]:
model = get_peft_model(model, lora_config)

In [24]:
model.print_trainable_parameters()

trainable params: 2,981,888 || all params: 1,002,867,840 || trainable%: 0.2973


In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"every_n_layers": 4},
    bf16=True,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [31]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

In [32]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
)

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [34]:
trainer.train()

# New Section